In [1]:
!pip install -q transformers torch pandas scikit-learn tqdm

In [2]:
import pandas as pd

# Load the dataset files
df_movies = pd.read_csv('movies_enriched.csv')
df_reviews = pd.read_csv('movies_with_reviews.csv')

# Merge datasets on 'movie_id' and check columns
df = pd.merge(df_movies, df_reviews, on='movie_id', how='left')

# View columns to confirm actual names after merging
print(df.columns.tolist())

# Fill missing text columns with empty strings safely
text_columns = ['title', 'genres', 'plot_summary', 'tagline', 'cast_bios', 'crew_bios', 'reviews']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

['movie_id', 'title_x', 'genres_x', 'plot_summary', 'tagline', 'cast_bios', 'crew_bios', 'title_y', 'genres_y', 'reviews', 'review_count']


In [3]:
combined_columns = []
for col in ['title', 'genres', 'plot_summary', 'tagline', 'cast_bios', 'crew_bios', 'reviews']:
    if col in df.columns:
        combined_columns.append(df[col])

df['combined_text'] = " ".join(combined_columns) if len(combined_columns) == 1 else combined_columns[0].copy()

for col in combined_columns[1:]:
    df['combined_text'] += " " + col

In [4]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()  # Set model to evaluation mode


C:\Users\KIIT0001\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [5]:
def get_mlm_embedding(text):
    encoded_input = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        output = model(**encoded_input)
    token_embeddings = output.last_hidden_state  # shape: [1, seq_len, hidden_dim]
    attention_mask = encoded_input['attention_mask']
    
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
    sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
    
    mean_embedding = sum_embeddings / sum_mask
    return mean_embedding.squeeze().cpu().numpy()


In [6]:
mlm_embeddings = []
for text in tqdm(df['combined_text'], desc='Extracting MLM embeddings'):
    embedding = get_mlm_embedding(text)
    mlm_embeddings.append(embedding)

mlm_embeddings = np.vstack(mlm_embeddings)
print("MLM embeddings shape:", mlm_embeddings.shape)


Extracting MLM embeddings:   0%|          | 0/3883 [00:00<?, ?it/s]Extracting MLM embeddings: 100%|██████████| 3883/3883 [1:16:08<00:00,  1.18s/it]


MLM embeddings shape: (3883, 768)


In [7]:
from sklearn.decomposition import PCA

pca = PCA(n_components=64)
mlm_embeddings_64 = pca.fit_transform(mlm_embeddings)
print("Reduced MLM embeddings shape:", mlm_embeddings_64.shape)


Reduced MLM embeddings shape: (3883, 64)


In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
mlm_embeddings_norm = scaler.fit_transform(mlm_embeddings_64)

print("Min normalized:", mlm_embeddings_norm.min())
print("Max normalized:", mlm_embeddings_norm.max())

np.save('movie_mlm_embeddings_64_normalized.npy', mlm_embeddings_norm)
print("Normalized embeddings saved.")


Min normalized: 0.0
Max normalized: 1.0000001
Normalized embeddings saved.


In [9]:
import pandas as pd

df_sample = pd.DataFrame(mlm_embeddings_norm[:5])
print(df_sample)


         0         1         2         3         4         5         6   \
0  0.960791  0.710521  0.647293  0.209224  0.487686  0.700823  0.614775   
1  0.951704  0.835659  0.580617  0.440415  0.549975  0.617029  0.552824   
2  0.946486  0.777774  0.534673  0.439054  0.545325  0.282446  0.425539   
3  0.936312  0.837921  0.624437  0.434331  0.800964  0.367792  0.637256   
4  0.952120  0.743330  0.688498  0.436099  0.633325  0.465771  0.511662   

         7         8         9   ...        54        55        56        57  \
0  0.567887  0.871657  0.546689  ...  0.515521  0.634293  0.607766  0.663645   
1  0.481927  0.736795  0.448436  ...  0.441286  0.564021  0.476519  0.678238   
2  0.665346  0.623234  0.303454  ...  0.322686  0.630185  0.491817  0.636479   
3  0.308424  0.673033  0.440166  ...  0.328058  0.501714  0.508930  0.654298   
4  0.446065  0.709500  0.451013  ...  0.462806  0.648893  0.515315  0.676972   

         58        59        60        61        62        63  
0  0